# Silver Layer Orchestrator – dim_pdv

**Purpose:**  
End-to-end orchestration of Silver layer for PDV dimension with explicit quality gates and monitoring. Ensures business-driven transformation, validation, and operational observability.

**Flow:**
1. **Transformation** (`run_transformation_dim_pdv`) – Cleans, standardizes, and enriches PDV data. Returns execution metrics (dict).
2. **Validation** (`run_validation_dim_pdv`) – Applies business-driven data quality rules. Returns metrics DataFrame (quality gate).
3. **Monitoring** (`run_monitoring_dim_pdv`) – Interprets validation metrics, evaluates health, and detects trends. Returns monitoring DataFrame (health check).

**Tables involved:**
- `workspace.silver.dim_pdv` (transformation output)
- `workspace.silver.validation_dim_pdv_results` (record-level validation)
- `workspace.silver.validation_dim_pdv_metrics` (aggregated validation metrics)
- `workspace.silver.monitoring_dim_pdv` (monitoring output)
- `workspace.silver.orchestration_log_dim_pdv` (orchestration log)

**Architecture:** Medallion, Databricks Serverless, Unity Catalog

**Quality Gates:**
- ✅ Transformation must return `status='SUCCESS'`
- ✅ Validation: `valid_percentage >= 95%` (configurable)
- ⚠️ Monitoring: Logs warning if `has_critical_alert=True`

**Exit Codes:**
- `0`: SUCCESS
- `1`: TRANSFORMATION_FAILED
- `2`: VALIDATION_FAILED (quality gate)
- `3`: MONITORING_FAILED

**Design Principles:**
- No business logic in orchestrator
- No .collect(), .count(), cache(), persist()
- No mutation of Silver data outside scripts
- All thresholds hardcoded for auditability (see script docstrings)
- All outputs are Delta tables, partitioned for performance

**References:**
- See transformation_dim_pdv.py, validation_dim_pdv.py, monitoring_dim_pdv.py for business logic and technical details.


In [ ]:
import logging
from datetime import datetime
import uuid
import json
from pyspark.sql import SparkSession, functions as F

# Configure logging
logger = logging.getLogger("silver_dim_pdv_orchestrator")
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("=" * 80)
logger.info("SILVER LAYER ORCHESTRATOR - dim_pdv")
logger.info("=" * 80)

In [ ]:
# Create widgets for configuration (Databricks only)
try:
    dbutils.widgets.text("quality_threshold", "95.0", "Min Valid %")
    dbutils.widgets.dropdown("fail_on_quality_gate", "true", ["true", "false"], "Fail on Quality Gate")
    
    QUALITY_THRESHOLD = float(dbutils.widgets.get("quality_threshold"))
    FAIL_ON_QUALITY_GATE = dbutils.widgets.get("fail_on_quality_gate").lower() == "true"
    
    logger.info(f"Configuration loaded from widgets:")
except:
    # Fallback for local/Jupyter execution
    QUALITY_THRESHOLD = 95.0
    FAIL_ON_QUALITY_GATE = True
    logger.info(f"Configuration using defaults (no widgets available):")

logger.info(f"  - quality_threshold: {QUALITY_THRESHOLD}%")
logger.info(f"  - fail_on_quality_gate: {FAIL_ON_QUALITY_GATE}")

In [ ]:
# Get or create Spark session
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()
if spark is None:
    logger.info("No active SparkSession found, creating new one...")
    spark = SparkSession.builder.appName("Silver_dim_pdv_orchestrator").getOrCreate()
else:
    logger.info("Using active SparkSession")

logger.info(f"Spark version: {spark.version}")

In [ ]:
# CELL 5: Import Orchestration Functions
import sys

# Adjust path for your environment
sys.path.append("../dimension_pdv/")

try:
    from transformation_dim_pdv import run_transformation_dim_pdv
    from validation_dim_pdv import run_validation_dim_pdv
    from monitoring_dim_pdv import run_monitoring_dim_pdv
    
    logger.info("✅ Successfully imported orchestration functions")
except ImportError as e:
    logger.error(f"❌ Failed to import: {e}")
    raise

In [ ]:
# Initialize execution context
run_id = str(uuid.uuid4())
execution_timestamp = datetime.utcnow()

logger.info("=" * 80)
logger.info(f"Orchestration Run ID: {run_id}")
logger.info(f"Execution Timestamp: {execution_timestamp.isoformat()}")
logger.info("=" * 80)

# Initialize summary dictionary
summary = {
    'run_id': run_id,
    'execution_timestamp': execution_timestamp.isoformat(),
    'transformation': {},
    'validation': {},
    'monitoring': {},
    'status': 'STARTED',
    'error_message': None,
    'exit_code': 0
}

# Align summary fields to transformation, validation, monitoring outputs
# Transformation: status, source_table, target_table, records_read, records_written, duration_seconds, execution_timestamp
# Validation: total_records, valid_records, valid_percentage, critical_violations, major_violations, minor_violations
# Monitoring: dataset_health_status, has_critical_alert, has_degraded_alert, has_trend_warning, monitoring_summary

In [ ]:
logger.info("")
logger.info("=" * 80)
logger.info("STEP 1/3: SILVER TRANSFORMATION")
logger.info("=" * 80)

try:
    # Execute transformation
    transformation_result = run_transformation_dim_pdv(spark=spark)
    
    # Store result
    summary['transformation'] = transformation_result
    
    # Log result
    logger.info(f"Transformation Status: {transformation_result['status']}")
    logger.info(f"Records Read: {transformation_result.get('records_read', 'N/A')}")
    logger.info(f"Records Written: {transformation_result.get('records_written', 'N/A')}")
    logger.info(f"Duration: {transformation_result.get('duration_seconds', 'N/A')}s")
    
    # Validate transformation succeeded
    if transformation_result['status'] != 'SUCCESS':
        error_msg = f"Transformation failed: {transformation_result.get('error_message', 'Unknown error')}"
        logger.error(f"❌ {error_msg}")
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 1
        raise Exception(error_msg)
    
    logger.info("✅ Transformation completed successfully")
    
except Exception as e:
    logger.error(f"❌ Transformation step failed: {str(e)}", exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 1
    raise

In [ ]:
logger.info("")
logger.info("=" * 80)
logger.info("STEP 2/3: SILVER VALIDATION")
logger.info("=" * 80)

try:
    # Execute validation
    validation_metrics_df = run_validation_dim_pdv(spark=spark)
    
    # Convert DataFrame to dict for summary
    validation_metrics = validation_metrics_df.first().asDict()
    
    # Store simplified metrics in summary (align to validation_dim_pdv.py)
    summary['validation'] = {
        'total_records': int(validation_metrics.get('total_records') or 0),
        'valid_records': int(validation_metrics.get('valid_records') or 0),
        'valid_percentage': float(validation_metrics.get('valid_percentage') or 0.0),
        'critical_violations': int(validation_metrics.get('critical_violations') or 0),
        'major_violations': int(validation_metrics.get('major_violations') or 0),
        'minor_violations': int(validation_metrics.get('minor_violations') or 0),
        'business_key_violations': int(validation_metrics.get('business_key_violations') or 0),
        'coordinates_violations': int(validation_metrics.get('coordinates_violations') or 0),
        'domain_violations': int(validation_metrics.get('domain_violations') or 0),
        'logic_violations': int(validation_metrics.get('logic_violations') or 0),
        'temporal_violations': int(validation_metrics.get('temporal_violations') or 0)
    }
    
    # Log validation results
    logger.info(f"Total Records: {summary['validation']['total_records']}")
    logger.info(f"Valid Records: {summary['validation']['valid_records']}")
    logger.info(f"Valid Percentage: {summary['validation']['valid_percentage']:.2f}%")
    logger.info(f"Critical Violations: {summary['validation']['critical_violations']}")
    logger.info(f"Major Violations: {summary['validation']['major_violations']}")
    logger.info(f"Minor Violations: {summary['validation']['minor_violations']}")
    logger.info(f"Business Key Violations: {summary['validation']['business_key_violations']}")
    logger.info(f"Coordinates Violations: {summary['validation']['coordinates_violations']}")
    logger.info(f"Domain Violations: {summary['validation']['domain_violations']}")
    logger.info(f"Logic Violations: {summary['validation']['logic_violations']}")
    logger.info(f"Temporal Violations: {summary['validation']['temporal_violations']}")
    
    # ===== QUALITY GATE =====
    logger.info("")
    logger.info("🚦 QUALITY GATE CHECK")
    logger.info(f"   Threshold: {QUALITY_THRESHOLD}%")
    logger.info(f"   Actual: {summary['validation']['valid_percentage']:.2f}%")
    
    if summary['validation']['valid_percentage'] < QUALITY_THRESHOLD:
        error_msg = f"Quality gate failed: {summary['validation']['valid_percentage']:.2f}% < {QUALITY_THRESHOLD}%"
        logger.error(f"❌ {error_msg}")
        
        if FAIL_ON_QUALITY_GATE:
            summary['status'] = 'FAILED'
            summary['error_message'] = error_msg
            summary['exit_code'] = 2
            raise Exception(error_msg)
        else:
            logger.warning("⚠️ Quality gate failed but continuing (fail_on_quality_gate=false)")
    else:
        logger.info("✅ Quality gate passed")
    
    logger.info("✅ Validation completed successfully")
    
except Exception as e:
    logger.error(f"❌ Validation step failed: {str(e)}", exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 2
    raise

In [ ]:
logger.info("")
logger.info("=" * 80)
logger.info("STEP 3/3: SILVER MONITORING")
logger.info("=" * 80)

try:
    # Execute monitoring
    monitoring_df = run_monitoring_dim_pdv(spark=spark)
    
    # Convert DataFrame to dict for summary
    monitoring_result = monitoring_df.first().asDict()
    
    # Defensive: replace None with 0 for int/float fields to avoid TypeError
    def safe_int(val):
        return int(val) if val is not None else 0
    def safe_float(val):
        return float(val) if val is not None else 0.0
    
    # Store simplified monitoring in summary (align to monitoring_dim_pdv.py)
    summary['monitoring'] = {
        'dataset_health_status': monitoring_result['dataset_health_status'],
        'has_critical_alert': bool(monitoring_result['has_critical_alert']),
        'has_degraded_alert': bool(monitoring_result.get('has_degraded_alert', False)),
        'has_business_key_alert': bool(monitoring_result.get('has_business_key_alert', False)),
        'has_coordinates_alert': bool(monitoring_result.get('has_coordinates_alert', False)),
        'has_domain_alert': bool(monitoring_result.get('has_domain_alert', False)),
        'has_logic_alert': bool(monitoring_result.get('has_logic_alert', False)),
        'has_temporal_alert': bool(monitoring_result.get('has_temporal_alert', False)),
        'detailed_alerts_count': safe_int(monitoring_result.get('detailed_alerts_count')),
        'has_trend_warning': bool(monitoring_result['has_trend_warning']),
        'valid_pct_ma7': safe_float(monitoring_result.get('valid_pct_ma7')),
        'critical_ma7': safe_float(monitoring_result.get('critical_ma7')),
        'monitoring_summary': monitoring_result['monitoring_summary']
    }
    
    # Log monitoring results
    logger.info(f"Health Status: {summary['monitoring']['dataset_health_status']}")
    logger.info(f"Critical Alert: {summary['monitoring']['has_critical_alert']}")
    logger.info(f"Degraded Alert: {summary['monitoring']['has_degraded_alert']}")
    logger.info(f"Business Key Alert: {summary['monitoring']['has_business_key_alert']}")
    logger.info(f"Coordinates Alert: {summary['monitoring']['has_coordinates_alert']}")
    logger.info(f"Domain Alert: {summary['monitoring']['has_domain_alert']}")
    logger.info(f"Logic Alert: {summary['monitoring']['has_logic_alert']}")
    logger.info(f"Temporal Alert: {summary['monitoring']['has_temporal_alert']}")
    logger.info(f"Detailed Alerts Count: {summary['monitoring']['detailed_alerts_count']}")
    logger.info(f"Trend Warning: {summary['monitoring']['has_trend_warning']}")
    logger.info(f"Valid % MA7: {summary['monitoring']['valid_pct_ma7']:.2f}")
    logger.info(f"Critical MA7: {summary['monitoring']['critical_ma7']:.2f}")
    logger.info(f"Summary: {summary['monitoring']['monitoring_summary']}")
    
    # Check for critical alerts
    if summary['monitoring']['has_critical_alert']:
        logger.warning("")
        logger.warning("⚠️" * 40)
        logger.warning(f"CRITICAL ALERT DETECTED: {summary['monitoring']['monitoring_summary']}")
        logger.warning("⚠️" * 40)
    
    logger.info("✅ Monitoring completed successfully")
    
except Exception as e:
    logger.error(f"❌ Monitoring step failed: {str(e)}", exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 3
    raise

In [ ]:
# Mark orchestration as successful if we got here
if summary['status'] == 'STARTED':
    summary['status'] = 'SUCCESS'
    summary['exit_code'] = 0

logger.info("")
logger.info("=" * 80)
logger.info(f"ORCHESTRATION COMPLETED: {summary['status']}")
logger.info("=" * 80)

# Persist orchestration log to Delta table
logger.info("Persisting orchestration log...")

try:
    orchestration_log = spark.createDataFrame([{ 
        'run_id': summary['run_id'],
        'execution_timestamp': summary['execution_timestamp'],
        'status': summary['status'],
        'exit_code': summary['exit_code'],
        'error_message': summary['error_message'],
        # Transformation fields
        'transformation_status': summary['transformation'].get('status'),
        'transformation_source_table': summary['transformation'].get('source_table'),
        'transformation_target_table': summary['transformation'].get('target_table'),
        'transformation_records_read': summary['transformation'].get('records_read'),
        'transformation_records_written': summary['transformation'].get('records_written'),
        'transformation_duration_seconds': summary['transformation'].get('duration_seconds'),
        # Validation fields
        'validation_total_records': summary['validation'].get('total_records'),
        'validation_valid_records': summary['validation'].get('valid_records'),
        'validation_valid_percentage': summary['validation'].get('valid_percentage'),
        'validation_critical_violations': summary['validation'].get('critical_violations'),
        'validation_major_violations': summary['validation'].get('major_violations'),
        'validation_minor_violations': summary['validation'].get('minor_violations'),
        'validation_business_key_violations': summary['validation'].get('business_key_violations'),
        'validation_coordinates_violations': summary['validation'].get('coordinates_violations'),
        'validation_domain_violations': summary['validation'].get('domain_violations'),
        'validation_logic_violations': summary['validation'].get('logic_violations'),
        'validation_temporal_violations': summary['validation'].get('temporal_violations'),
        # Monitoring fields
        'monitoring_health_status': summary['monitoring'].get('dataset_health_status'),
        'monitoring_has_critical_alert': summary['monitoring'].get('has_critical_alert'),
        'monitoring_has_degraded_alert': summary['monitoring'].get('has_degraded_alert'),
        'monitoring_has_business_key_alert': summary['monitoring'].get('has_business_key_alert'),
        'monitoring_has_coordinates_alert': summary['monitoring'].get('has_coordinates_alert'),
        'monitoring_has_domain_alert': summary['monitoring'].get('has_domain_alert'),
        'monitoring_has_logic_alert': summary['monitoring'].get('has_logic_alert'),
        'monitoring_has_temporal_alert': summary['monitoring'].get('has_temporal_alert'),
        'monitoring_detailed_alerts_count': summary['monitoring'].get('detailed_alerts_count'),
        'monitoring_has_trend_warning': summary['monitoring'].get('has_trend_warning'),
        'monitoring_valid_pct_ma7': summary['monitoring'].get('valid_pct_ma7'),
        'monitoring_critical_ma7': summary['monitoring'].get('critical_ma7'),
        'monitoring_summary': summary['monitoring'].get('monitoring_summary'),
        # Full summary
        'full_summary_json': json.dumps(summary, default=str)
    }])
    
    # Write to orchestration log table
    orchestration_log.write.format("delta").mode("append").saveAsTable("workspace.silver.orchestration_log_dim_pdv")
    
    logger.info("✅ Orchestration log persisted successfully")
except Exception as e:
    logger.warning(f"⚠️ Failed to persist orchestration log: {e}")

In [ ]:
import pprint

print("\n" + "=" * 80)
print("ORCHESTRATION SUMMARY")
print("=" * 80)
pprint.pprint(summary, width=120, compact=False)
print("=" * 80)

# Display key metrics (aligned to scripts)
print(f"\n📊 KEY METRICS:")
print(f"   Status: {summary['status']}")
print(f"   Exit Code: {summary['exit_code']}")
print(f"   Records Processed: {summary['transformation'].get('records_written', 'N/A')}")
print(f"   Data Quality: {summary['validation'].get('valid_percentage', 'N/A'):.2f}%")
print(f"   Health Status: {summary['monitoring'].get('dataset_health_status', 'N/A')}")
print(f"   Critical Alert: {summary['monitoring'].get('has_critical_alert', 'N/A')}")
print(f"   Business Key Violations: {summary['validation'].get('business_key_violations', 'N/A')}")
print(f"   Coordinates Violations: {summary['validation'].get('coordinates_violations', 'N/A')}")
print(f"   Domain Violations: {summary['validation'].get('domain_violations', 'N/A')}")
print(f"   Logic Violations: {summary['validation'].get('logic_violations', 'N/A')}")
print(f"   Temporal Violations: {summary['validation'].get('temporal_violations', 'N/A')}")
print(f"   Monitoring Summary: {summary['monitoring'].get('monitoring_summary', 'N/A')}")

In [ ]:
# For Databricks Workflows: Exit with status
try:
    exit_payload = {
        'status': summary['status'],
        'exit_code': summary['exit_code'],
        'run_id': summary['run_id'],
        'error_message': summary['error_message'],
        'valid_percentage': summary['validation'].get('valid_percentage'),
        'critical_violations': summary['validation'].get('critical_violations'),
        'major_violations': summary['validation'].get('major_violations'),
        'minor_violations': summary['validation'].get('minor_violations'),
        'business_key_violations': summary['validation'].get('business_key_violations'),
        'coordinates_violations': summary['validation'].get('coordinates_violations'),
        'domain_violations': summary['validation'].get('domain_violations'),
        'logic_violations': summary['validation'].get('logic_violations'),
        'temporal_violations': summary['validation'].get('temporal_violations'),
        'health_status': summary['monitoring'].get('dataset_health_status'),
        'has_critical_alert': summary['monitoring'].get('has_critical_alert'),
        'monitoring_summary': summary['monitoring'].get('monitoring_summary')
    }
    
    logger.info(f"Exiting with payload: {exit_payload}")
    dbutils.notebook.exit(json.dumps(exit_payload))
    
except NameError:
    # Not in Databricks environment (Jupyter)
    logger.info("Not in Databricks environment, skipping dbutils.notebook.exit()")
    
    if summary['status'] == 'FAILED':
        logger.error(f"Orchestration failed with exit code {summary['exit_code']}")
        raise Exception(f"Orchestration failed: {summary['error_message']}")
    else:
        logger.info("✅ Orchestration completed successfully in Jupyter environment")